# 🧬 Dark Manifold v7 — Environment-Coupled Dynamics

## Core Insight

A cell doesn't exist in a vacuum. It:
- **Consumes** nutrients from the environment (glucose, O2, amino acids)
- **Secretes** waste products (lactate, CO2)
- **Senses** environmental changes and adapts metabolism
- **Changes** its environment by consuming/secreting

This creates a **feedback loop** that forces realistic dynamics!

## Why v6 Failed

v6 had no environment pressure. The model found a "lazy" solution:
barely move anything → satisfy stability → done.

## v7 Solution

Environment depletes → cell MUST metabolize → creates real dynamics

In [ ]:
#@title 1️⃣ Setup
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
import matplotlib.pyplot as plt

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data (same as v6)

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}
name_to_idx = {n.lower(): i for i, n in enumerate(met_names)}

# Parse genes
genes = set()
gene_to_rxns = {}
for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        if name.lower() in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name.lower() in left else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Find key metabolite indices
def find_met_idx(keyword):
    for i, n in enumerate(met_names):
        if keyword in n.lower():
            return i
    return 0

atp_idx = find_met_idx('atp')
adp_idx = find_met_idx('adp')
amp_idx = find_met_idx('amp')
glucose_idx = find_met_idx('d-glucose')
pyruvate_idx = find_met_idx('pyruvate')
lactate_idx = find_met_idx('lactate')

print(f"Genes: {n_genes}, Metabolites: {n_mets}, Reactions: {n_rxns}")
print(f"\nKey indices:")
print(f"  ATP: {atp_idx}, ADP: {adp_idx}, AMP: {amp_idx}")
print(f"  Glucose: {glucose_idx}, Pyruvate: {pyruvate_idx}, Lactate: {lactate_idx}")

In [ ]:
#@title 3️⃣ Environment State Definition

# Environment variables
ENV_VARS = [
    'glucose_ext',    # External glucose (mM)
    'lactate_ext',    # External lactate (mM)
    'oxygen',         # Oxygen availability (0-1)
    'amino_acids',    # Amino acid pool (mM)
    'ph',             # pH (6.0-8.0)
    'temperature',    # Temperature factor (0.8-1.2)
]
n_env = len(ENV_VARS)

# Initial environment (rich medium)
RICH_MEDIUM = {
    'glucose_ext': 10.0,    # 10 mM glucose
    'lactate_ext': 0.0,     # No lactate initially
    'oxygen': 1.0,          # Aerobic
    'amino_acids': 5.0,     # Amino acids available
    'ph': 7.0,              # Neutral pH
    'temperature': 1.0,     # Normal temperature
}

# Starvation environment
STARVATION = {
    'glucose_ext': 0.1,
    'lactate_ext': 0.0,
    'oxygen': 1.0,
    'amino_acids': 0.5,
    'ph': 7.0,
    'temperature': 1.0,
}

def env_to_tensor(env_dict, device='cpu'):
    return torch.tensor([env_dict[k] for k in ENV_VARS], dtype=torch.float32, device=device)

print(f"Environment variables: {n_env}")
print(f"Rich medium: {RICH_MEDIUM}")

In [ ]:
#@title 4️⃣ DarkManifoldV7 - Environment-Coupled Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class DarkManifoldV7(nn.Module):
    """
    Environment-Coupled Cellular Model.
    
    The cell and environment are a coupled dynamical system:
    - Cell consumes nutrients → environment depletes
    - Cell secretes waste → environment changes (pH, lactate)
    - Environment sensed → cell adapts gene expression
    """
    
    def __init__(self, n_genes, n_metabolites, n_reactions, n_env, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_metabolites
        self.n_rxns = n_reactions
        self.n_env = n_env
        
        # Fixed biochemistry
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # Substrate mask
        self.register_buffer('substrate_mask', (self.S < 0).float())
        
        # ============ ENVIRONMENT SENSING ============
        # Sensor network: environment → gene regulation
        self.env_sensor = nn.Sequential(
            nn.Linear(n_env, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_genes),
            nn.Tanh(),  # Regulation in [-1, 1]
        )
        
        # ============ TRANSPORT ============
        # Glucose transporter (uptake)
        self.glucose_Km = nn.Parameter(torch.tensor(0.5))  # mM
        self.glucose_Vmax = nn.Parameter(torch.tensor(1.0))  # mM/step
        
        # Lactate exporter (secretion)
        self.lactate_Km = nn.Parameter(torch.tensor(1.0))
        self.lactate_Vmax = nn.Parameter(torch.tensor(0.5))
        
        # ============ INTERNAL METABOLISM ============
        # Gene → Enzyme
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
            nn.Softplus(),
        )
        
        # Metabolite encoder
        self.met_encoder = nn.Sequential(
            nn.Linear(n_metabolites, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
        )
        
        # Km and tau
        self.log_Km = nn.Parameter(torch.zeros(n_reactions))
        self.log_tau = nn.Parameter(torch.zeros(n_reactions))
        
        # ============ BIOMASS & GROWTH ============
        # Growth rate predictor based on metabolic state
        self.growth_predictor = nn.Sequential(
            nn.Linear(n_metabolites + n_env, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),  # Growth rate 0-1
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.1, 10.0)
    
    def transport(self, met_conc, env_state, atp_idx, glucose_idx, lactate_idx):
        """
        Compute transport fluxes between cell and environment.
        """
        batch_size = met_conc.shape[0]
        
        # Get concentrations
        glucose_ext = env_state[:, 0]  # glucose_ext is first env var
        glucose_int = met_conc[:, glucose_idx]
        lactate_int = met_conc[:, lactate_idx]
        atp = met_conc[:, atp_idx]
        
        # Glucose uptake (Michaelis-Menten, ATP-dependent)
        glucose_Km = torch.exp(self.glucose_Km).clamp(0.01, 10.0)
        glucose_Vmax = torch.exp(self.glucose_Vmax).clamp(0.01, 5.0)
        
        # Uptake depends on external glucose and ATP availability
        atp_factor = atp / (0.5 + atp)  # ATP-dependent
        glucose_uptake = glucose_Vmax * glucose_ext / (glucose_Km + glucose_ext) * atp_factor
        glucose_uptake = glucose_uptake.clamp(min=0, max=glucose_ext)  # Can't uptake more than available
        
        # Lactate export (concentration-driven)
        lactate_Km = torch.exp(self.lactate_Km).clamp(0.01, 10.0)
        lactate_Vmax = torch.exp(self.lactate_Vmax).clamp(0.01, 5.0)
        lactate_export = lactate_Vmax * lactate_int / (lactate_Km + lactate_int)
        lactate_export = lactate_export.clamp(min=0, max=lactate_int)  # Can't export more than have
        
        return glucose_uptake, lactate_export
    
    def forward(self, gene_expr, met_conc, env_state, 
                atp_idx, adp_idx, glucose_idx, lactate_idx, dt=0.01):
        """
        One step of coupled cell-environment dynamics.
        """
        batch_size = gene_expr.shape[0]
        
        # ===== 1. SENSE ENVIRONMENT =====
        # Environment affects gene expression
        env_signal = self.env_sensor(env_state)  # (B, n_genes)
        regulated_genes = gene_expr * (1.0 + 0.5 * env_signal)  # Modulate expression
        regulated_genes = regulated_genes.clamp(min=0.1)
        
        # ===== 2. TRANSPORT =====
        glucose_uptake, lactate_export = self.transport(
            met_conc, env_state, atp_idx, glucose_idx, lactate_idx
        )
        
        # ===== 3. INTERNAL METABOLISM =====
        # Gene → Enzyme
        Vmax = self.gene_encoder(regulated_genes)
        
        # Substrate concentrations
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
        substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
        
        # Michaelis-Menten
        mm_term = substrate_conc / (self.Km + substrate_conc)
        
        # Flux modulation from metabolite state
        flux_mod = torch.sigmoid(self.met_encoder(met_conc))
        
        # Combined flux
        flux = Vmax * mm_term * flux_mod * self.tau
        
        # Stoichiometry
        dM_dt = flux @ self.S.T
        
        # ===== 4. UPDATE CELL STATE =====
        met_next = met_conc + dt * dM_dt
        
        # Add transport effects
        met_next[:, glucose_idx] = met_next[:, glucose_idx] + glucose_uptake
        met_next[:, lactate_idx] = met_next[:, lactate_idx] - lactate_export
        # ATP cost of transport
        met_next[:, atp_idx] = met_next[:, atp_idx] - 0.1 * glucose_uptake
        met_next[:, adp_idx] = met_next[:, adp_idx] + 0.1 * glucose_uptake
        
        met_next = met_next.clamp(min=0)
        
        # ===== 5. UPDATE ENVIRONMENT =====
        env_next = env_state.clone()
        env_next[:, 0] = (env_state[:, 0] - glucose_uptake).clamp(min=0)  # glucose_ext
        env_next[:, 1] = env_state[:, 1] + lactate_export  # lactate_ext
        
        # pH drops with lactate accumulation
        env_next[:, 4] = 7.0 - 0.1 * env_next[:, 1].clamp(max=5.0)  # pH
        
        # ===== 6. GROWTH RATE =====
        growth_input = torch.cat([met_next, env_next], dim=-1)
        growth_rate = self.growth_predictor(growth_input).squeeze(-1)
        
        return {
            'met_next': met_next,
            'env_next': env_next,
            'flux': flux,
            'glucose_uptake': glucose_uptake,
            'lactate_export': lactate_export,
            'growth_rate': growth_rate,
            'regulated_genes': regulated_genes,
        }
    
    def simulate(self, gene_expr, met_init, env_init, n_steps,
                 atp_idx, adp_idx, glucose_idx, lactate_idx, dt=0.01):
        """
        Run coupled simulation.
        """
        met_traj = [met_init]
        env_traj = [env_init]
        growth_traj = []
        uptake_traj = []
        
        met = met_init
        env = env_init
        
        for _ in range(n_steps):
            out = self.forward(
                gene_expr, met, env,
                atp_idx, adp_idx, glucose_idx, lactate_idx, dt
            )
            met = out['met_next']
            env = out['env_next']
            
            met_traj.append(met)
            env_traj.append(env)
            growth_traj.append(out['growth_rate'])
            uptake_traj.append(out['glucose_uptake'])
        
        return {
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
            'growth_rates': torch.stack(growth_traj, dim=1),
            'uptake_rates': torch.stack(uptake_traj, dim=1),
        }


model = DarkManifoldV7(
    n_genes=n_genes,
    n_metabolites=n_mets,
    n_reactions=n_rxns,
    n_env=n_env,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV7 (Environment-Coupled):")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Environment variables: {n_env}")

In [ ]:
#@title 5️⃣ Environment-Aware Loss Functions

class EnvironmentLoss(nn.Module):
    """
    Losses that encourage realistic cell-environment dynamics.
    """
    
    def __init__(self, atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx):
        super().__init__()
        self.atp_idx = atp_idx
        self.adp_idx = adp_idx
        self.amp_idx = amp_idx
        self.glucose_idx = glucose_idx
        self.lactate_idx = lactate_idx
    
    def survival_loss(self, met_traj, env_traj):
        """
        Cell should survive: maintain ATP and energy charge.
        """
        # ATP should stay above threshold
        atp = met_traj[:, :, self.atp_idx]
        atp_penalty = F.relu(1.0 - atp).mean()  # Penalize ATP < 1mM
        
        # Energy charge should stay healthy
        adp = met_traj[:, :, self.adp_idx]
        amp = met_traj[:, :, self.amp_idx]
        ec = (atp + 0.5 * adp) / (atp + adp + amp + 1e-6)
        ec_penalty = F.relu(0.7 - ec).mean()  # Penalize EC < 0.7
        
        return atp_penalty + ec_penalty
    
    def activity_loss(self, uptake_rates, growth_rates):
        """
        Cell should be DOING something - consuming glucose, growing.
        Penalize inactivity.
        """
        # Should have non-zero uptake when glucose available
        uptake_penalty = F.relu(0.1 - uptake_rates).mean()
        
        # Should have non-zero growth when nutrients available
        growth_penalty = F.relu(0.1 - growth_rates).mean()
        
        return uptake_penalty + growth_penalty
    
    def glucose_depletion_loss(self, env_traj):
        """
        Glucose should decrease over time (being consumed).
        """
        glucose_ext = env_traj[:, :, 0]  # (B, T)
        
        # Glucose at end should be less than start
        start = glucose_ext[:, 0]
        end = glucose_ext[:, -1]
        
        # Penalize if not depleting
        depletion = start - end
        depletion_penalty = F.relu(0.5 - depletion).mean()  # Should deplete at least 0.5 mM
        
        return depletion_penalty
    
    def lactate_production_loss(self, env_traj):
        """
        Lactate should increase (fermentation product).
        """
        lactate_ext = env_traj[:, :, 1]
        
        start = lactate_ext[:, 0]
        end = lactate_ext[:, -1]
        
        production = end - start
        production_penalty = F.relu(0.1 - production).mean()  # Should produce some lactate
        
        return production_penalty
    
    def stability_loss(self, met_traj):
        """
        No explosion or collapse.
        """
        # No negative concentrations
        neg_penalty = F.relu(-met_traj).mean()
        
        # No explosions
        explosion_penalty = F.relu(met_traj - 100).mean()
        
        return neg_penalty + explosion_penalty
    
    def forward(self, met_traj, env_traj, uptake_rates, growth_rates):
        """
        Combined loss.
        """
        loss_survive = self.survival_loss(met_traj, env_traj)
        loss_activity = self.activity_loss(uptake_rates, growth_rates)
        loss_glucose = self.glucose_depletion_loss(env_traj)
        loss_lactate = self.lactate_production_loss(env_traj)
        loss_stable = self.stability_loss(met_traj)
        
        total = (
            5.0 * loss_survive +
            3.0 * loss_activity +
            2.0 * loss_glucose +
            1.0 * loss_lactate +
            10.0 * loss_stable
        )
        
        return total, {
            'survive': loss_survive.item(),
            'activity': loss_activity.item(),
            'glucose': loss_glucose.item(),
            'lactate': loss_lactate.item(),
            'stable': loss_stable.item(),
        }


loss_fn = EnvironmentLoss(atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx)
print("✅ EnvironmentLoss defined")

In [ ]:
#@title 6️⃣ Train with Environmental Scenarios

n_epochs = 2000
batch_size = 32
n_steps = 100
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

# Initial cell state
met_init = torch.ones(n_mets, device=device) * 0.5
met_init[atp_idx] = 4.0
met_init[adp_idx] = 0.5
met_init[amp_idx] = 0.1
met_init[glucose_idx] = 0.5
met_init[lactate_idx] = 0.1

def train_step():
    model.train()
    
    # Random initial conditions
    gene_expr = torch.rand(batch_size, n_genes, device=device) * 1.5 + 0.5
    met = met_init.unsqueeze(0).expand(batch_size, -1).clone()
    met = met * (0.8 + 0.4 * torch.rand(batch_size, n_mets, device=device))
    
    # Random environment (vary glucose level)
    env = torch.zeros(batch_size, n_env, device=device)
    env[:, 0] = torch.rand(batch_size, device=device) * 10 + 2  # glucose 2-12 mM
    env[:, 1] = 0.0  # lactate starts at 0
    env[:, 2] = 1.0  # oxygen
    env[:, 3] = 5.0  # amino acids
    env[:, 4] = 7.0  # pH
    env[:, 5] = 1.0  # temperature
    
    # Run simulation
    result = model.simulate(
        gene_expr, met, env, n_steps,
        atp_idx, adp_idx, glucose_idx, lactate_idx
    )
    
    # Compute loss
    loss, metrics = loss_fn(
        result['met_trajectory'],
        result['env_trajectory'],
        result['uptake_rates'],
        result['growth_rates'],
    )
    
    return loss, metrics

losses = []
print(f"Training v7: {n_epochs} epochs")
print(f"Environment-coupled dynamics with survival pressure")
print()

for epoch in tqdm(range(n_epochs)):
    optimizer.zero_grad()
    loss, metrics = train_step()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f} | "
              f"survive={metrics['survive']:.3f} activity={metrics['activity']:.3f} "
              f"glucose={metrics['glucose']:.3f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
#@title 7️⃣ Evaluate: Glucose Depletion Scenario

model.eval()

# Rich medium scenario
gene_expr = torch.ones(1, n_genes, device=device)
met = met_init.unsqueeze(0).clone()
env = env_to_tensor(RICH_MEDIUM, device).unsqueeze(0)

with torch.no_grad():
    result = model.simulate(
        gene_expr, met, env, n_steps=300,
        atp_idx=atp_idx, adp_idx=adp_idx,
        glucose_idx=glucose_idx, lactate_idx=lactate_idx
    )

met_traj = result['met_trajectory'][0].cpu().numpy()
env_traj = result['env_trajectory'][0].cpu().numpy()
growth_rates = result['growth_rates'][0].cpu().numpy()
uptake_rates = result['uptake_rates'][0].cpu().numpy()

# Plot
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Environment: Glucose depletion
axes[0, 0].plot(env_traj[:, 0], 'b-', linewidth=2)
axes[0, 0].set_title('External Glucose')
axes[0, 0].set_xlabel('Time step')
axes[0, 0].set_ylabel('mM')
axes[0, 0].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Low threshold')

# Environment: Lactate accumulation
axes[0, 1].plot(env_traj[:, 1], 'orange', linewidth=2)
axes[0, 1].set_title('External Lactate')
axes[0, 1].set_xlabel('Time step')
axes[0, 1].set_ylabel('mM')

# Environment: pH
axes[0, 2].plot(env_traj[:, 4], 'purple', linewidth=2)
axes[0, 2].set_title('pH')
axes[0, 2].set_xlabel('Time step')
axes[0, 2].axhline(y=6.5, color='r', linestyle='--', alpha=0.5, label='Stress threshold')

# Cell: ATP
axes[1, 0].plot(met_traj[:, atp_idx], 'g-', linewidth=2, label='ATP')
axes[1, 0].plot(met_traj[:, adp_idx], 'y-', linewidth=2, label='ADP')
axes[1, 0].set_title('Energy Metabolites')
axes[1, 0].set_xlabel('Time step')
axes[1, 0].set_ylabel('mM')
axes[1, 0].legend()

# Growth rate over time
axes[1, 1].plot(growth_rates, 'teal', linewidth=2)
axes[1, 1].set_title('Growth Rate')
axes[1, 1].set_xlabel('Time step')
axes[1, 1].set_ylabel('Rate')

# Uptake rate over time
axes[1, 2].plot(uptake_rates, 'coral', linewidth=2)
axes[1, 2].set_title('Glucose Uptake Rate')
axes[1, 2].set_xlabel('Time step')
axes[1, 2].set_ylabel('mM/step')

plt.suptitle('v7: Environment-Coupled Dynamics (Rich Medium → Depletion)', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v7.png', dpi=150)
plt.show()

print(f"\n=== SUMMARY ===")
print(f"Glucose: {env_traj[0, 0]:.2f} → {env_traj[-1, 0]:.2f} mM (depleted: {env_traj[0, 0] - env_traj[-1, 0]:.2f})")
print(f"Lactate: {env_traj[0, 1]:.2f} → {env_traj[-1, 1]:.2f} mM (produced: {env_traj[-1, 1] - env_traj[0, 1]:.2f})")
print(f"pH: {env_traj[0, 4]:.2f} → {env_traj[-1, 4]:.2f}")
print(f"ATP: {met_traj[0, atp_idx]:.2f} → {met_traj[-1, atp_idx]:.2f} mM")

In [ ]:
#@title 8️⃣ Gene Knockout Under Nutrient Stress

print("="*60)
print("KNOCKOUT ANALYSIS UNDER NUTRIENT STRESS")
print("="*60)

# Test knockouts in LIMITED glucose environment
gene_wt = torch.ones(1, n_genes, device=device)
met = met_init.unsqueeze(0).clone()

# Limited glucose environment
env_limited = env_to_tensor({
    'glucose_ext': 3.0,  # Lower glucose
    'lactate_ext': 0.0,
    'oxygen': 1.0,
    'amino_acids': 2.0,
    'ph': 7.0,
    'temperature': 1.0,
}, device).unsqueeze(0)

# Wildtype
with torch.no_grad():
    wt_result = model.simulate(
        gene_wt, met, env_limited, n_steps=200,
        atp_idx=atp_idx, adp_idx=adp_idx,
        glucose_idx=glucose_idx, lactate_idx=lactate_idx
    )

wt_atp_final = wt_result['met_trajectory'][0, -1, atp_idx].item()
wt_growth_avg = wt_result['growth_rates'].mean().item()

print(f"Wildtype: ATP={wt_atp_final:.2f}, Growth={wt_growth_avg:.3f}")

# Test each knockout
ko_results = []
for i in tqdm(range(n_genes)):
    gene_ko = gene_wt.clone()
    gene_ko[0, i] = 0.0
    
    with torch.no_grad():
        ko_result = model.simulate(
            gene_ko, met.clone(), env_limited.clone(), n_steps=200,
            atp_idx=atp_idx, adp_idx=adp_idx,
            glucose_idx=glucose_idx, lactate_idx=lactate_idx
        )
    
    ko_atp = ko_result['met_trajectory'][0, -1, atp_idx].item()
    ko_growth = ko_result['growth_rates'].mean().item()
    
    # Lethal if ATP < 50% or growth < 50%
    is_lethal = (ko_atp < 0.5 * wt_atp_final) or (ko_growth < 0.3 * wt_growth_avg)
    
    ko_results.append({
        'gene': genes[i],
        'atp': ko_atp,
        'growth': ko_growth,
        'atp_ratio': ko_atp / (wt_atp_final + 1e-6),
        'growth_ratio': ko_growth / (wt_growth_avg + 1e-6),
        'is_lethal': is_lethal,
    })

# Summary
lethal = [r for r in ko_results if r['is_lethal']]
print(f"\nResults:")
print(f"  Essential: {len(lethal)}/{n_genes} ({100*len(lethal)/n_genes:.1f}%)")
print(f"  Non-essential: {n_genes - len(lethal)}/{n_genes}")

# Most lethal
sorted_ko = sorted(ko_results, key=lambda x: x['growth_ratio'])
print(f"\nMost lethal knockouts:")
for r in sorted_ko[:10]:
    status = "❌" if r['is_lethal'] else "✓"
    print(f"  {status} {r['gene']}: growth={r['growth_ratio']:.0%} ATP={r['atp_ratio']:.0%}")

In [ ]:
#@title 9️⃣ Save

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'env_vars': ENV_VARS,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'knockout_results': ko_results,
    'version': 'v7_environment_coupled',
    'approach': 'cell_environment_feedback_loop',
    'metrics': {
        'n_essential': len(lethal),
        'wt_atp': wt_atp_final,
        'wt_growth': wt_growth_avg,
        'glucose_depleted': env_traj[0, 0] - env_traj[-1, 0],
        'lactate_produced': env_traj[-1, 1] - env_traj[0, 1],
    }
}

torch.save(save_dict, 'dark_manifold_v7_env.pt')

print("="*60)
print("DARK MANIFOLD v7 - SUMMARY")
print("="*60)
print(f"\nApproach: Cell-environment feedback loop")
print(f"\nEnvironment coupling:")
print(f"  - Glucose uptake: cell consumes, environment depletes")
print(f"  - Lactate secretion: cell produces, environment acidifies")
print(f"  - Sensing: environment state modulates gene expression")
print(f"\nDynamics:")
print(f"  - Glucose depleted: {env_traj[0, 0] - env_traj[-1, 0]:.2f} mM")
print(f"  - Lactate produced: {env_traj[-1, 1] - env_traj[0, 1]:.2f} mM")
print(f"\nKnockout results:")
print(f"  - Essential genes: {len(lethal)}/{n_genes} ({100*len(lethal)/n_genes:.1f}%)")

from google.colab import files
files.download('dark_manifold_v7_env.pt')
files.download('trajectory_v7.png')